## Assignment 4

<br>

### Question 1
Investigate the model for predicting Diabetes disease progression by adding more explanatory variables to it in addition to `bmi` and `s5`.

a) Which variable would you add next? Why?

b) How does adding it affect the model's performance? Compute metrics and compare to having just `bmi` and `s5`.

d) Does it help if you add even more variables?

Include your own findings and explanations in code comments or inside triple quotes """...""".

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

print(df.head())

        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  


In [2]:
correlations = df.corr()["target"].sort_values(ascending=False)
print(correlations)

target    1.000000
bmi       0.586450
s5        0.565883
bp        0.441482
s4        0.430453
s6        0.382483
s1        0.212022
age       0.187889
s2        0.174054
sex       0.043062
s3       -0.394789
Name: target, dtype: float64


In [3]:
y = df['target']
x1 = df[['bmi', 's5']]
model1 = LinearRegression()
model1.fit(x1, y)

y_pred1 = model1.predict(x1)
r2_m1 = r2_score(y, y_pred1)
mse_m1 = mean_squared_error(y, y_pred1)

print("\n--- Old model (bmi + s5) ---")
print(f"R2 Score: {r2_m1:.4f}")
print(f"MSE: {mse_m1:.4f}")


--- Old model (bmi + s5) ---
R2 Score: 0.4595
MSE: 3205.1901


In [4]:
y = df['target']
chosen_variable = 'bp'
x2 = df[['bmi', 's5', chosen_variable]]
model2 = LinearRegression()
model2.fit(x2, y)

y_pred2 = model2.predict(x2)
r2_m2 = r2_score(y, y_pred2)
mse_m2 = mean_squared_error(y, y_pred2)

print(f"\n--- New model (bmi + s5 + {chosen_variable}) ---")
print(f"R2 Score: {r2_m2:.4f}")
print(f"MSE: {mse_m2:.4f}")


--- New model (bmi + s5 + bp) ---
R2 Score: 0.4801
MSE: 3083.0513


Summarization Question (1)
* a) The next variable add is bp, because it has the highest Correlation with the target.
* b) After adding bp improves the model's performance:
The R-squared score increased from **0.4595** to **0.4801**.
The MSE decreased from **3205.1901** to **3083.0513**.
* d) To adding too many variables might increase the R-squared score,but it is risky. It can cause overfitting.

### Question 2

Consider the dataset `50_Startups.csv` which contains data for companies' profit etc.

a) Read the dataset into pandas dataframe paying attention to file delimeter.

b) Identify the variables inside the dataset

c) Investigate the correlation between the variables

d) Choose appropriate variables to predict company profit. Justify your choice.

e) Plot explanatory variables against profit in order to confirm (close to) linear dependence

f) Form training and testing data (80/20 split)

g) Train linear regression model with training data

h) Compute RMSE and $R^2$ values for training and testing data separately

Include your own findings and explanations in code comments or inside triple quotes """...""".

Answer A > Import CSV file

In [6]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv('50_Startups.csv', delimiter=',')

Answer B > Show the dataset and top 5 data header

In [7]:
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     str    
 4   Profit           50 non-null     float64
dtypes: float64(4), str(1)
memory usage: 2.1 KB
None
   R&D Spend  Administration  Marketing Spend       State     Profit
0  165349.20       136897.80        471784.10    New York  192261.83
1  162597.70       151377.59        443898.53  California  191792.06
2  153441.51       101145.55        407934.54     Florida  191050.39
3  144372.41       118671.85        383199.62    New York  182901.99
4  142107.34        91391.77        366168.42     Florida  166187.94


Answer C > Import CSV file


In [8]:
numeric_cols = df.select_dtypes(include=[np.number])
corr_matrix = numeric_cols.corr()
print(corr_matrix)

                 R&D Spend  Administration  Marketing Spend    Profit
R&D Spend         1.000000        0.241955         0.724248  0.972900
Administration    0.241955        1.000000        -0.032154  0.200717
Marketing Spend   0.724248       -0.032154         1.000000  0.747766
Profit            0.972900        0.200717         0.747766  1.000000


In [10]:
features = ['R&D Spend', 'Marketing Spend']
x = df[features]
y = df['Profit']

In [13]:
plt.figure(figsize=(12, 5))

for i, col in enumerate(features, 1):
    plt.subplot(1, len(features), i)
    sns.scatterplot(data=df, x=col, y='Profit')
    plt.title(f'{col} vs Profit')
    plt.grid(True)

plt.tight_layout()
plt.show()

TypeError: 'module' object is not callable

### Question 3

Consider car performance data from the file `Auto.csv`.

a) Read the data into pandas dataframe

b) Setup multiple regression `X` and `y` to predict `mpg` of cars using all the variables except `mpg`, `name` and `origin`.

c) Split data into training and testing sets (80/20 split)

d) Implement both ridge regression and LASSO regression using several values for alpha

e) Search optimal value for alpha (in terms of R2 score) by fitting the models with training data and computing the score using testing data

f) Plot the R2 scores for both regressors as functions of alpha

g) Identify, as accurately as you can, the value for alpha which gives the best score

 
Include your own findings and explanations in code comments or inside triple quotes """...""".